# AQI Model Retraining for API Integration

This notebook retrains the AQI prediction model to match the features created by the API's `create_features()` function, ensuring proper integration between the trained model and the Flask API.

## Goals:
1. Analyze current data and API feature engineering
2. Train a new model that matches API expectations  
3. Save updated model artifacts
4. Validate API integration

## Step 1: Import Libraries and Load Data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
# Load the training data
df = pd.read_csv('../data/aqi_data.csv')

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nBasic Statistics:")
print(df.describe())

## Step 2: Define Feature Engineering (Matching API)

In [ ]:
# Define the create_features function (copied from app.py)
def create_features(data):
    """Create features from input data - matches API implementation"""
    features = {
        'PM2.5': data['PM2.5'],
        'PM10': data['PM10'],
        'NO2': data['NO2'],
        'SO2': data['SO2'],
        'CO': data['CO'],
        'OZONE': data['OZONE'],
        'NH3': data.get('NH3', 0),
        'PM_ratio': data['PM2.5'] / (data['PM10'] + 1),
        'NOx_SOx_ratio': data['NO2'] / (data['SO2'] + 1),
        'PM_NO2_interaction': data['PM2.5'] * data['NO2'],
        'PM_CO_interaction': data['PM2.5'] * data['CO'],
        'NO2_SO2_interaction': data['NO2'] * data['SO2'],
        'Total_PM': data['PM2.5'] + data['PM10'],
        'Total_Gas': data['NO2'] + data['SO2'] + data['CO'],
        'latitude': data.get('latitude', 28.6),
        'longitude': data.get('longitude', 77.2)
    }

    # For training, we'll use simple encoding (can be improved later)
    # Since we don't have encoders yet, use dummy values
    features['state_encoded'] = 0  # Will be replaced with proper encoding
    features['city_encoded'] = 0   # Will be replaced with proper encoding

    return features

# Test the function
sample_data = df.iloc[0]
features = create_features(sample_data)
print("Features created by API function:")
for key, value in features.items():
    print(f"  {key}: {value}")

## Step 3: Prepare Training Data

In [ ]:
# Prepare training data by applying feature engineering to all rows
feature_list = []
targets = []

for idx, row in df.iterrows():
    try:
        features = create_features(row)
        feature_list.append(features)
        targets.append(row['AQI'])
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
        continue

# Convert to DataFrame
X = pd.DataFrame(feature_list)
y = pd.Series(targets)

print(f"Training data shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")
print(f"\nFirst 5 rows of features:")
print(X.head())
print(f"\nTarget statistics:")
print(y.describe())

## Step 4: Train New Model

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest model (similar to original)
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

print("Training model...")
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

# Evaluate
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)
train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)

print("
Model Performance:")
print(f"Train RMSE: {train_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")
print(f"Train R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")
print(f"Train MAE: {train_mae:.2f}")
print(f"Test MAE: {test_mae:.2f}")

## Step 5: Save Model Artifacts

In [ ]:
# Save the new model and artifacts
import os

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save model
with open('../models/best_aqi_model_new.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save scaler
with open('../models/scaler_new.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save feature names
feature_names = list(X.columns)
with open('../models/feature_names_new.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

# Save metadata
metadata = {
    'model_type': 'RandomForestRegressor',
    'n_estimators': 200,
    'max_depth': 20,
    'features': feature_names,
    'train_rmse': train_rmse,
    'test_rmse': test_rmse,
    'train_r2': train_r2,
    'test_r2': test_r2,
    'training_date': pd.Timestamp.now().isoformat(),
    'data_shape': X.shape,
    'api_compatible': True
}

with open('../models/metadata_new.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print("✓ New model artifacts saved:")
print("  - models/best_aqi_model_new.pkl")
print("  - models/scaler_new.pkl") 
print("  - models/feature_names_new.pkl")
print("  - models/metadata_new.pkl")

print(f"\nModel expects {len(feature_names)} features:")
for i, feat in enumerate(feature_names, 1):
    print(f"  {i}. {feat}")

## Step 6: Validate API Integration

In [ ]:
# Test that the new model works with API-style input
test_sample = {
    'pm25': 50.0,
    'pm10': 100.0,
    'no2': 30.0,
    'so2': 15.0,
    'co': 1.5,
    'ozone': 40.0,
    'nh3': 5.0,
    'city': 'Delhi',
    'state': 'Delhi',
    'latitude': 28.6,
    'longitude': 77.2
}

# Create features using API function
api_features = create_features({
    'PM2.5': test_sample['pm25'],
    'PM10': test_sample['pm10'],
    'NO2': test_sample['no2'],
    'SO2': test_sample['so2'],
    'CO': test_sample['co'],
    'OZONE': test_sample['ozone'],
    'NH3': test_sample['nh3'],
    'latitude': test_sample['latitude'],
    'longitude': test_sample['longitude']
})

# Convert to DataFrame and scale
api_X = pd.DataFrame([api_features])
api_X_scaled = scaler.transform(api_X)

# Predict
prediction = model.predict(api_X_scaled)[0]

print("API Integration Test:")
print(f"Input: {test_sample}")
print(f"Features: {api_features}")
print(f"Prediction: {prediction:.1f} AQI")

# Compare with actual from dataset (find similar sample)
similar_samples = df[
    (df['PM2.5'].between(45, 55)) & 
    (df['PM10'].between(95, 105)) &
    (df['NO2'].between(25, 35))
]

if len(similar_samples) > 0:
    actual_aqi = similar_samples['AQI'].mean()
    print(f"Similar actual AQI: {actual_aqi:.1f}")
    print(f"Difference: {abs(prediction - actual_aqi):.1f}")
else:
    print("No similar samples found for comparison")

print("\n✓ API integration validation complete!")

## Next Steps

1. **Update app.py**: Replace the old model files with the new ones (_new.pkl)
2. **Test API**: Run the Flask app and test predictions
3. **Enhance Data**: Add weather data and more features as per Part 1 guide
4. **Advanced Models**: Implement LSTM and ensemble models from Part 2
5. **Deploy**: Add dashboard and monitoring from Part 3

## Files to Update in app.py:
- Change `'models/best_aqi_model.pkl'` to `'models/best_aqi_model_new.pkl'`
- Change `'models/scaler.pkl'` to `'models/scaler_new.pkl'`
- Change `'models/feature_names.pkl'` to `'models/feature_names_new.pkl'`

The new model is now compatible with the API's feature engineering!